---
## 7. Config A — Hyperparameter Optimisation

The model-free agents trained in Sections 3 and 4 used default hyperparameters and only 50,000 episodes. Looking at the learning curves, both agents were still improving at the end of training — meaning they had not fully converged yet.

In this section we systematically investigate two questions:

1. **Does more training help?** We run both SARSA and Q-Learning with 50k, 200k, and 500k episodes.
2. **What is the best learning rate alpha?** We run a grid search over alpha ∈ {0.05, 0.1, 0.2, 0.3, 0.5} for **both** algorithms, keeping everything else fixed.

Crucially, **both algorithms are tuned under exactly the same conditions** — same episode counts, same alpha candidates, same seed. This ensures the final comparison is fair: any difference in performance is due to the algorithm itself, not different tuning.

Beyond raw survival rate, we also compare **policy stability** — the standard deviation of returns across evaluation episodes. In a real ICU, an agent that is slightly less optimal but much more consistent is arguably preferable to one with a higher average but catastrophic failures on some patients.


### 7.1 — Effect of Number of Training Episodes

We test both SARSA and Q-Learning at 50k (already done), 200k, and 500k episodes, keeping alpha=0.3 fixed. This isolates the effect of training length from the effect of the learning rate.


In [ ]:
# ── Effect of training length: SARSA and Q-Learning ──────────────────────────
# 50k results already exist from Sections 3 and 4
# We extend to 200k and 500k for both algorithms

episode_counts = [200_000, 500_000]

# Store results: dict keyed by (algorithm, n_episodes)
ep_results = {
    ('SARSA',      50_000): sarsa_results,
    ('Q-Learning', 50_000): ql_results,
}
ep_returns = {
    ('SARSA',      50_000): sarsa_returns,
    ('Q-Learning', 50_000): ql_returns,
}

for n_ep in episode_counts:
    for algo_name, algo_fn in [('SARSA', sarsa), ('Q-Learning', q_learning)]:
        print(f'Training {algo_name} | alpha=0.3 | {n_ep:,} episodes...')
        Q_tmp, returns_tmp = algo_fn(
            n_episodes=n_ep,
            alpha=0.3,
            gamma=GAMMA,
            epsilon_start=1.0,
            epsilon_min=0.01,
            seed=SEED,
        )
        policy_tmp  = np.argmax(Q_tmp, axis=1)
        results_tmp = evaluate_policy_tabular(policy_tmp)
        ep_results[(algo_name, n_ep)]  = results_tmp
        ep_returns[(algo_name, n_ep)]  = returns_tmp
        print(f'  Survival rate : {results_tmp["survival_rate"]:.1f}%  |  '
              f'Mean return: {results_tmp["mean_return"]:.4f}')

print('\nDone.')


In [ ]:
# ── Plot: survival rate vs episodes for both algorithms ───────────────────────

n_ep_list   = [50_000, 200_000, 500_000]
ep_labels   = ['50k', '200k', '500k']

sarsa_ep_survival = [ep_results[('SARSA',      n)]['survival_rate'] for n in n_ep_list]
ql_ep_survival    = [ep_results[('Q-Learning', n)]['survival_rate'] for n in n_ep_list]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: grouped bar chart
x     = np.arange(len(ep_labels))
width = 0.35

bars_s = axes[0].bar(x - width/2, sarsa_ep_survival, width,
                     label='SARSA', color='steelblue', alpha=0.85, edgecolor='white')
bars_q = axes[0].bar(x + width/2, ql_ep_survival, width,
                     label='Q-Learning', color='darkorange', alpha=0.85, edgecolor='white')

axes[0].axhline(pi_results['survival_rate'], color='tomato', linestyle='--',
                linewidth=1.5, label=f'PI ceiling ({pi_results["survival_rate"]:.1f}%)')
axes[0].axhline(survival_rate, color='gray', linestyle=':',
                linewidth=1.5, label=f'Random ({survival_rate:.1f}%)')
axes[0].set_xticks(x)
axes[0].set_xticklabels(ep_labels)
axes[0].set_xlabel('Training episodes')
axes[0].set_ylabel('Survival Rate (%)')
axes[0].set_title('SARSA vs Q-Learning: Survival Rate vs Training Length\n(alpha=0.3 fixed)',
                  fontweight='bold')
axes[0].set_ylim(60, 85)
axes[0].legend(fontsize=9)

for bar, val in zip(list(bars_s) + list(bars_q),
                    sarsa_ep_survival + ql_ep_survival):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Right: learning curves at 500k for both
window = 2000
for algo, color in [('SARSA', 'steelblue'), ('Q-Learning', 'darkorange')]:
    roll = pd.Series(ep_returns[(algo, 500_000)]).rolling(window).mean()
    axes[1].plot(roll, color=color, linewidth=2.0, label=f'{algo} (500k)')

axes[1].axhline(pi_results['mean_return'], color='tomato', linestyle='--',
                linewidth=1.5, label='PI ceiling')
axes[1].axhline(np.mean(rand_returns), color='gray', linestyle=':',
                linewidth=1.5, label='Random baseline')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Rolling mean return')
axes[1].set_title('Learning Curves at 500k Episodes\n(alpha=0.3 fixed)', fontweight='bold')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/configA_episode_count_comparison.png', bbox_inches='tight')
plt.show()

print('Observation:')
print('If Q-Learning has a consistently higher bar: its off-policy updates')
print('converge to a better policy given the same data.')
print('If SARSA closes the gap with more episodes: its conservative updates')
print('just need more time to accumulate the right experience.')


### 7.2 — Alpha Grid Search: SARSA and Q-Learning

We now run the full alpha grid search for **both** algorithms under identical conditions:
- Alpha values: {0.05, 0.1, 0.2, 0.3, 0.5}
- 200k episodes each (good quality without excessive compute)
- Same epsilon decay, same gamma, same seed

This lets us ask: do SARSA and Q-Learning prefer the same learning rate? An on-policy algorithm like SARSA may prefer a smaller, more cautious alpha because its updates include the noise of exploratory actions. Q-Learning's off-policy updates are slightly more stable, so it may tolerate a larger alpha.


In [ ]:
# ── Alpha grid search: SARSA and Q-Learning, 200k episodes ───────────────────

alpha_values = [0.05, 0.1, 0.2, 0.3, 0.5]

# Results dicts keyed by (algorithm, alpha)
alpha_grid_results = {}
alpha_grid_returns = {}

for algo_name, algo_fn in [('SARSA', sarsa), ('Q-Learning', q_learning)]:
    print(f'\n── {algo_name} alpha grid search ──')
    for alpha_val in alpha_values:
        print(f'  alpha={alpha_val} | 200k episodes...')
        Q_tmp, returns_tmp = algo_fn(
            n_episodes=200_000,
            alpha=alpha_val,
            gamma=GAMMA,
            epsilon_start=1.0,
            epsilon_min=0.01,
            seed=SEED,
        )
        policy_tmp  = np.argmax(Q_tmp, axis=1)
        results_tmp = evaluate_policy_tabular(policy_tmp)
        alpha_grid_results[(algo_name, alpha_val)] = results_tmp
        alpha_grid_returns[(algo_name, alpha_val)] = returns_tmp
        print(f'    Survival: {results_tmp["survival_rate"]:.1f}%  '
              f'Return: {results_tmp["mean_return"]:.4f}')

# Find best alpha per algorithm
best_alpha_sarsa = max(alpha_values,
    key=lambda a: alpha_grid_results[('SARSA', a)]['survival_rate'])
best_alpha_ql    = max(alpha_values,
    key=lambda a: alpha_grid_results[('Q-Learning', a)]['survival_rate'])

print(f'\nBest alpha — SARSA      : {best_alpha_sarsa} '
      f'({alpha_grid_results[("SARSA", best_alpha_sarsa)]["survival_rate"]:.1f}%)')
print(f'Best alpha — Q-Learning : {best_alpha_ql} '
      f'({alpha_grid_results[("Q-Learning", best_alpha_ql)]["survival_rate"]:.1f}%)')


In [ ]:
# ── Plot: alpha grid search results for both algorithms ───────────────────────

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for row, (algo, color, best_a) in enumerate([
    ('SARSA',      'steelblue',  best_alpha_sarsa),
    ('Q-Learning', 'darkorange', best_alpha_ql),
]):
    survival_by_alpha = [alpha_grid_results[(algo, a)]['survival_rate']
                         for a in alpha_values]

    # Left column: bar chart survival vs alpha
    bar_colors = [color if a == best_a else '#d6eaf8' if algo == 'SARSA'
                  else '#fdebd0' for a in alpha_values]
    bars = axes[row, 0].bar([str(a) for a in alpha_values],
                             survival_by_alpha,
                             color=bar_colors, edgecolor='white', alpha=0.9)
    axes[row, 0].axhline(pi_results['survival_rate'], color='tomato',
                         linestyle='--', linewidth=1.5,
                         label=f'PI ceiling ({pi_results["survival_rate"]:.1f}%)')
    axes[row, 0].axhline(survival_rate, color='gray', linestyle=':',
                         linewidth=1.5, label=f'Random ({survival_rate:.1f}%)')
    axes[row, 0].set_xlabel('Learning rate alpha')
    axes[row, 0].set_ylabel('Survival Rate (%)')
    axes[row, 0].set_title(f'{algo}: Survival Rate vs Alpha\n(200k episodes, highlighted = best)',
                            fontweight='bold')
    axes[row, 0].set_ylim(60, 85)
    axes[row, 0].legend(fontsize=8)
    for bar, val in zip(bars, survival_by_alpha):
        axes[row, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                          f'{val:.1f}%', ha='center', va='bottom',
                          fontsize=9, fontweight='bold')

    # Right column: learning curves for all alpha values
    cmap = plt.cm.Blues if algo == 'SARSA' else plt.cm.Oranges
    window = 1000
    for i, a_val in enumerate(alpha_values):
        c   = cmap(0.35 + i * 0.13)
        lw  = 2.5 if a_val == best_a else 1.0
        lab = f'alpha={a_val}' + (' ← best' if a_val == best_a else '')
        roll = pd.Series(alpha_grid_returns[(algo, a_val)]).rolling(window).mean()
        axes[row, 1].plot(roll, color=c, linewidth=lw, label=lab)

    axes[row, 1].axhline(pi_results['mean_return'], color='tomato',
                         linestyle='--', linewidth=1.5, label='PI ceiling')
    axes[row, 1].axhline(np.mean(rand_returns), color='gray',
                         linestyle=':', linewidth=1.5, label='Random baseline')
    axes[row, 1].set_xlabel('Episode')
    axes[row, 1].set_ylabel('Rolling mean return')
    axes[row, 1].set_title(f'{algo}: Learning Curves by Alpha (200k episodes)',
                            fontweight='bold')
    axes[row, 1].legend(fontsize=7)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/configA_alpha_grid_both.png', bbox_inches='tight')
plt.show()

print(f'SARSA best alpha      : {best_alpha_sarsa}')
print(f'Q-Learning best alpha : {best_alpha_ql}')
print()
print('Do they prefer different alphas?')
if best_alpha_sarsa < best_alpha_ql:
    print('Yes — SARSA prefers a smaller alpha, consistent with its on-policy')
    print('nature: conservative updates help when learning from noisy exploratory actions.')
elif best_alpha_sarsa > best_alpha_ql:
    print('Interestingly, SARSA prefers a larger alpha here.')
    print('This may reflect the specific reward structure of this environment.')
else:
    print('Both algorithms prefer the same alpha — the reward structure')
    print('of this environment may dominate over the on/off-policy distinction.')


### 7.3. Policy Stability: SARSA vs Q-Learning

Raw survival rate is not the only metric that matters. In a clinical setting, **consistency is as important as average performance**. An agent that saves 75% of patients reliably is preferable to one that saves 77% on average but occasionally drops to 60% — those bad episodes represent real patients dying.

We measure stability as the **standard deviation of episode returns** during evaluation. A lower standard deviation means the policy is more predictable and reliable.


In [ ]:
# ── Policy stability evaluation: SARSA vs Q-Learning ─────────────────────────
# Re-evaluate both optimised policies collecting full return distributions

def evaluate_policy_with_distribution(policy_array, n_episodes=2000, seed=SEED):
    """
    Evaluate a tabular policy and return the full return distribution.
    Returns dict with mean, std, survival rate, and the full returns array.
    """
    np.random.seed(seed)
    env_eval = make_sepsis_env()
    returns, lengths = [], []

    for _ in range(n_episodes):
        obs, _ = env_eval.reset(seed=np.random.randint(100_000))
        total_r, steps, done = 0.0, 0, False
        while not done:
            action = int(policy_array[int(obs)])
            obs, r, te, tr, _ = env_eval.step(action)
            total_r += r
            steps   += 1
            done     = te or tr
        returns.append(total_r)
        lengths.append(steps)

    env_eval.close()
    returns = np.array(returns)
    return {
        'mean_return'   : float(np.mean(returns)),
        'std_return'    : float(np.std(returns)),
        'survival_rate' : float(np.mean(returns > 0)) * 100,
        'mean_length'   : float(np.mean(lengths)),
        'returns_array' : returns,
    }


# Train optimised versions with best alpha per algorithm, 500k episodes
print(f'Training OPTIMISED SARSA | alpha={best_alpha_sarsa} | 500k episodes...')
sarsa_opt_Q, sarsa_opt_returns = sarsa(
    n_episodes=500_000,
    alpha=best_alpha_sarsa,
    gamma=GAMMA,
    epsilon_start=1.0,
    epsilon_min=0.01,
    seed=SEED,
)
sarsa_opt_policy = np.argmax(sarsa_opt_Q, axis=1)
sarsa_opt_dist   = evaluate_policy_with_distribution(sarsa_opt_policy)

print(f'  Survival: {sarsa_opt_dist["survival_rate"]:.1f}%  '
      f'Std: {sarsa_opt_dist["std_return"]:.4f}')

print(f'\nTraining OPTIMISED Q-Learning | alpha={best_alpha_ql} | 500k episodes...')
ql_opt_Q, ql_opt_returns = q_learning(
    n_episodes=500_000,
    alpha=best_alpha_ql,
    gamma=GAMMA,
    epsilon_start=1.0,
    epsilon_min=0.01,
    seed=SEED,
)
ql_opt_policy = np.argmax(ql_opt_Q, axis=1)
ql_opt_dist   = evaluate_policy_with_distribution(ql_opt_policy)

print(f'  Survival: {ql_opt_dist["survival_rate"]:.1f}%  '
      f'Std: {ql_opt_dist["std_return"]:.4f}')

# Also get distributions for PI and random for comparison
pi_dist   = evaluate_policy_with_distribution(pi_policy)
rand_dist_full = {'mean_return': float(np.mean(rand_returns)),
                  'std_return' : float(np.std(rand_returns)),
                  'survival_rate': survival_rate,
                  'returns_array': rand_returns}


In [ ]:
# ── Plot: return distributions and stability comparison ───────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Left: return distribution histograms
dist_configs = [
    ('Random Baseline',     rand_dist_full,  'gray'),
    ('Policy Iteration',    pi_dist,         'tomato'),
    ('SARSA (optimised)',   sarsa_opt_dist,  'steelblue'),
    ('Q-Learning (optimised)', ql_opt_dist, 'darkorange'),
]

for name, d, color in dist_configs:
    axes[0].hist(d['returns_array'], bins=30, alpha=0.45,
                 color=color, label=name, density=True, edgecolor='none')

axes[0].set_xlabel('Episode Return')
axes[0].set_ylabel('Density')
axes[0].set_title('Return Distributions\n(2000 evaluation episodes)', fontweight='bold')
axes[0].legend(fontsize=8)

# Middle: mean ± std bars (reliability plot)
names  = ['Random', 'PI', 'SARSA\n(opt)', 'Q-Learning\n(opt)']
means  = [d['mean_return']   for _, d, _ in dist_configs]
stds   = [d['std_return']    for _, d, _ in dist_configs]
colors = [c                  for _, _, c in dist_configs]

axes[1].bar(names, means, color=colors, alpha=0.8, edgecolor='white')
axes[1].errorbar(names, means, yerr=stds, fmt='none',
                 color='black', capsize=5, linewidth=1.5)
axes[1].set_ylabel('Mean Return ± Std')
axes[1].set_title('Mean Return with Variability\n(lower std = more reliable)', fontweight='bold')
for i, (m, s) in enumerate(zip(means, stds)):
    axes[1].text(i, m + s + 0.01, f'σ={s:.3f}',
                 ha='center', fontsize=8, color='black')

# Right: survival rate comparison — original vs optimised
comparison_names = [
    'Random\nBaseline',
    'PI\nCeiling',
    'SARSA\nOriginal',
    'SARSA\nOptimised',
    'Q-Lrn\nOriginal',
    'Q-Lrn\nOptimised',
]
comparison_values = [
    survival_rate,
    pi_results['survival_rate'],
    sarsa_results['survival_rate'],
    sarsa_opt_dist['survival_rate'],
    ql_results['survival_rate'],
    ql_opt_dist['survival_rate'],
]
comp_colors = ['gray', 'tomato',
               '#aed6f1', 'steelblue',
               '#fad7a0', 'darkorange']

bars = axes[2].bar(comparison_names, comparison_values,
                   color=comp_colors, alpha=0.9, edgecolor='white')
axes[2].set_ylabel('Survival Rate (%)')
axes[2].set_title('Survival Rate: Original vs Optimised\n(light = original, dark = optimised)',
                  fontweight='bold')
axes[2].set_ylim(60, 85)
for bar, val in zip(bars, comparison_values):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/configA_stability_comparison.png', bbox_inches='tight')
plt.show()


### 7.4 — Full Optimisation Summary


In [ ]:
# ── Full comparison table ─────────────────────────────────────────────────────

def get_std(dist_or_results, returns_arr=None):
    """Helper to get std whether from dist dict or plain results dict."""
    if 'std_return' in dist_or_results:
        return dist_or_results['std_return']
    if returns_arr is not None:
        return float(np.std(returns_arr))
    return float('nan')


final_table = pd.DataFrame([
    {
        'Algorithm'          : 'Random Baseline',
        'Version'            : '—',
        'Episodes'           : '—',
        'Alpha'              : '—',
        'Survival Rate %'    : round(survival_rate, 1),
        'Mean Return'        : round(float(np.mean(rand_returns)), 4),
        'Std Return'         : round(float(np.std(rand_returns)), 4),
        'Gap to PI (pp)'     : round(pi_results['survival_rate'] - survival_rate, 1),
    },
    {
        'Algorithm'          : 'Policy Iteration',
        'Version'            : 'Optimal ceiling',
        'Episodes'           : '—',
        'Alpha'              : '—',
        'Survival Rate %'    : round(pi_dist['survival_rate'], 1),
        'Mean Return'        : round(pi_dist['mean_return'], 4),
        'Std Return'         : round(pi_dist['std_return'], 4),
        'Gap to PI (pp)'     : 0.0,
    },
    {
        'Algorithm'          : 'SARSA',
        'Version'            : 'Original',
        'Episodes'           : '50k',
        'Alpha'              : '0.3',
        'Survival Rate %'    : round(sarsa_results['survival_rate'], 1),
        'Mean Return'        : round(sarsa_results['mean_return'], 4),
        'Std Return'         : round(float(np.std(sarsa_returns[-1000:])), 4),
        'Gap to PI (pp)'     : round(pi_dist['survival_rate'] - sarsa_results['survival_rate'], 1),
    },
    {
        'Algorithm'          : 'SARSA',
        'Version'            : 'Optimised',
        'Episodes'           : '500k',
        'Alpha'              : str(best_alpha_sarsa),
        'Survival Rate %'    : round(sarsa_opt_dist['survival_rate'], 1),
        'Mean Return'        : round(sarsa_opt_dist['mean_return'], 4),
        'Std Return'         : round(sarsa_opt_dist['std_return'], 4),
        'Gap to PI (pp)'     : round(pi_dist['survival_rate'] - sarsa_opt_dist['survival_rate'], 1),
    },
    {
        'Algorithm'          : 'Q-Learning',
        'Version'            : 'Original',
        'Episodes'           : '50k',
        'Alpha'              : '0.3',
        'Survival Rate %'    : round(ql_results['survival_rate'], 1),
        'Mean Return'        : round(ql_results['mean_return'], 4),
        'Std Return'         : round(float(np.std(ql_returns[-1000:])), 4),
        'Gap to PI (pp)'     : round(pi_dist['survival_rate'] - ql_results['survival_rate'], 1),
    },
    {
        'Algorithm'          : 'Q-Learning',
        'Version'            : 'Optimised',
        'Episodes'           : '500k',
        'Alpha'              : str(best_alpha_ql),
        'Survival Rate %'    : round(ql_opt_dist['survival_rate'], 1),
        'Mean Return'        : round(ql_opt_dist['mean_return'], 4),
        'Std Return'         : round(ql_opt_dist['std_return'], 4),
        'Gap to PI (pp)'     : round(pi_dist['survival_rate'] - ql_opt_dist['survival_rate'], 1),
    },
    {
        'Algorithm'          : 'Double Q-Learning',
        'Version'            : 'Original',
        'Episodes'           : '50k',
        'Alpha'              : '0.3',
        'Survival Rate %'    : round(dql_results['survival_rate'], 1),
        'Mean Return'        : round(dql_results['mean_return'], 4),
        'Std Return'         : round(float(np.std(dql_returns[-1000:])), 4),
        'Gap to PI (pp)'     : round(pi_dist['survival_rate'] - dql_results['survival_rate'], 1),
    },
])

display(final_table)
final_table.to_csv(f'{PLOTS_DIR}/configA_full_optimisation_table.csv', index=False)


In [ ]:
# ── Final printed summary ─────────────────────────────────────────────────────

print('=' * 72)
print('CONFIG A — OPTIMISATION SUMMARY')
print('=' * 72)
print(f'{"Algorithm":<22} {"Version":<16} {"Survival %":>12} {"Std":>8} {"Gap to PI":>10}')
print('-' * 72)

rows = [
    ('Random Baseline',   '—',               survival_rate,                    float(np.std(rand_returns)),    pi_dist['survival_rate'] - survival_rate),
    ('Policy Iteration',  'Ceiling',          pi_dist['survival_rate'],         pi_dist['std_return'],         0.0),
    ('SARSA',             'Original (50k)',   sarsa_results['survival_rate'],   float(np.std(sarsa_returns[-1000:])), pi_dist['survival_rate'] - sarsa_results['survival_rate']),
    ('SARSA',             f'Optimised (500k)',sarsa_opt_dist['survival_rate'],  sarsa_opt_dist['std_return'],  pi_dist['survival_rate'] - sarsa_opt_dist['survival_rate']),
    ('Q-Learning',        'Original (50k)',   ql_results['survival_rate'],      float(np.std(ql_returns[-1000:])),   pi_dist['survival_rate'] - ql_results['survival_rate']),
    ('Q-Learning',        f'Optimised (500k)',ql_opt_dist['survival_rate'],     ql_opt_dist['std_return'],     pi_dist['survival_rate'] - ql_opt_dist['survival_rate']),
    ('Double Q-Learning', 'Original (50k)',   dql_results['survival_rate'],     float(np.std(dql_returns[-1000:])),  pi_dist['survival_rate'] - dql_results['survival_rate']),
]

for name, version, sr, std, gap in rows:
    print(f'{name:<22} {version:<16} {sr:>11.1f}% {std:>8.4f} {gap:>9.1f}pp')

print('=' * 72)
print()

sarsa_improvement = sarsa_opt_dist['survival_rate'] - sarsa_results['survival_rate']
ql_improvement    = ql_opt_dist['survival_rate']    - ql_results['survival_rate']

print('Improvement from optimisation:')
print(f'  SARSA:      +{sarsa_improvement:.1f}pp survival rate')
print(f'  Q-Learning: +{ql_improvement:.1f}pp survival rate')
print()
print('Stability (lower std = more reliable policy):')
sarsa_more_stable = sarsa_opt_dist['std_return'] < ql_opt_dist['std_return']
if sarsa_more_stable:
    print(f'  SARSA is more stable (std={sarsa_opt_dist["std_return"]:.4f}) '
          f'than Q-Learning (std={ql_opt_dist["std_return"]:.4f})')
    print('  This supports SARSA as the preferred choice in a clinical context')
    print('  where consistency matters as much as average survival rate.')
else:
    print(f'  Q-Learning is more stable (std={ql_opt_dist["std_return"]:.4f}) '
          f'than SARSA (std={sarsa_opt_dist["std_return"]:.4f})')
    print('  Q-Learning dominates on both survival rate and consistency.')
print()
print('Remaining gap to PI ceiling:')
print(f'  SARSA optimised:      {pi_dist["survival_rate"] - sarsa_opt_dist["survival_rate"]:.1f}pp')
print(f'  Q-Learning optimised: {pi_dist["survival_rate"] - ql_opt_dist["survival_rate"]:.1f}pp')
print()
print('This remaining gap represents the irreducible cost of not having')
print('the full MDP model. A model-free agent cannot fully close it')
print('without perfect environment knowledge, no matter how long it trains.')
